# Day 5 — 基本面数据抓取 (yfinance)

**目标**: 获取全股票池的季度财务数据 (自动从Prices.csv读取股票列表)  
**数据源**: Yahoo Finance (via yfinance)  
**输出**: 季度资产负债表 / 利润表 / 现金流 + 当前估值指标

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import time
import os

OUT_DIR = r"C:\Users\1\Desktop\项目\stock-data"

# 从Prices_fixed.csv自动读取股票列表 (适配扩充后的股票池)
prices_path = os.path.join(OUT_DIR, "Prices_fixed.csv")
df_prices = pd.read_csv(prices_path)
SYMBOLS = [c for c in df_prices.columns if c != "Date"]

print(f"目标股票 ({len(SYMBOLS)} 只): {SYMBOLS}")

In [ ]:
# ==========================================
# 第1步: 抓取季度财报 + 关键指标 (含重试)
# ==========================================

all_balance = {}    # {symbol: DataFrame}
all_income = {}     # {symbol: DataFrame}
all_cashflow = {}   # {symbol: DataFrame}
all_info = {}       # {symbol: dict}
errors = []

for i, sym in enumerate(SYMBOLS):
    print(f"[{i+1:2d}/{len(SYMBOLS)}] {sym:5s} ...", end=" ")
    try:
        ticker = yf.Ticker(sym)
        
        # 季度财报
        all_balance[sym] = ticker.quarterly_balance_sheet
        time.sleep(0.5)
        all_income[sym] = ticker.quarterly_financials
        time.sleep(0.5)
        all_cashflow[sym] = ticker.quarterly_cashflow
        time.sleep(0.5)
        
        # 当前估值指标
        all_info[sym] = ticker.info
        
        n_bs = len(all_balance[sym].columns) if all_balance[sym] is not None else 0
        n_is = len(all_income[sym].columns) if all_income[sym] is not None else 0
        print(f"OK (BS={n_bs}q, IS={n_is}q)")
    except Exception as e:
        errors.append(sym)
        print(f"ERR: {str(e)[:80]}")
    
    time.sleep(1.0)  # 限速

print(f"\n成功: {len(all_info)}/{len(SYMBOLS)}")
if errors:
    print(f"失败: {errors}")

In [ ]:
# ==========================================
# 第2步: 将季度财报转为长表 (便于后续merge)
# ==========================================

def stack_quarterly(sheet_dict, value_name):
    """将 {symbol: wide_df} 转为长表 DataFrame"""
    frames = []
    for sym, df_wide in sheet_dict.items():
        if df_wide is None or df_wide.empty:
            continue
        # df_wide: rows=指标, columns=日期
        df_long = df_wide.stack().reset_index()
        df_long.columns = ["item", "report_date", value_name]
        df_long["symbol"] = sym
        df_long["report_date"] = pd.to_datetime(df_long["report_date"])
        frames.append(df_long)
    return pd.concat(frames, ignore_index=True)

print("转换财报为长表...")

df_bs_long = stack_quarterly(all_balance, "balance_value")
print(f"资产负债表长表: {df_bs_long.shape}")

df_is_long = stack_quarterly(all_income, "income_value")
print(f"利润表长表: {df_is_long.shape}")

df_cf_long = stack_quarterly(all_cashflow, "cashflow_value")
print(f"现金流长表: {df_cf_long.shape}")

In [ ]:
# ==========================================
# 第3步: 透视 — 每只股票每个季度的关键指标
# ==========================================

# 每个sheet有哪些指标?
bs_items = df_bs_long["item"].unique()
is_items = df_is_long["item"].unique()
cf_items = df_cf_long["item"].unique()

print(f"资产负债表指标: {len(bs_items)} 个")
print(bs_items[:20].tolist())
print(f"\n利润表指标: {len(is_items)} 个")
print(is_items[:20].tolist())
print(f"\n现金流指标: {len(cf_items)} 个")
print(cf_items[:20].tolist())

In [ ]:
# ==========================================
# 第4步: 提取关键比率指标 (从info)
# ==========================================

info_rows = []
key_fields = [
    "symbol", "sector", "industry", "marketCap", "enterpriseValue",
    "trailingPE", "forwardPE", "priceToBook", "priceToSales",
    "bookValue", "earningsPerShare", "revenuePerShare",
    "returnOnEquity", "returnOnAssets", "profitMargins",
    "grossMargins", "operatingMargins", "ebitdaMargins",
    "debtToEquity", "currentRatio", "quickRatio",
    "revenueGrowth", "earningsGrowth", "earningsQuarterlyGrowth",
    "freeCashflow", "operatingCashflow", "totalCash", "totalDebt",
    "totalRevenue", "grossProfits", "ebitda",
    "beta", "52WeekChange", "heldPercentInstitutions",
    "shortPercentOfFloat", "shortRatio", "pegRatio"
]

for sym, info in all_info.items():
    row = {"symbol": sym}
    for f in key_fields:
        row[f] = info.get(f, np.nan)
    info_rows.append(row)

df_info = pd.DataFrame(info_rows)
print(f"Info维度: {df_info.shape}")
df_info.T

In [ ]:
# ==========================================
# 第5步: 季度财报透视 — 每只股票每个季度一行
# ==========================================

# 资产负债表关键指标
bs_key = [
    "Total Assets", "Total Current Assets", "Total Liabilities Net Minority Interest",
    "Total Equity Gross Minority Interest", "Working Capital",
    "Cash And Cash Equivalents", "Long Term Debt", "Current Debt",
    "Invested Capital", "Tangible Book Value",
    "Common Stock Equity", "Capital Stock",
]

# 利润表关键指标
is_key = [
    "Total Revenue", "Operating Revenue", "Gross Profit",
    "Operating Income", "EBIT", "EBITDA",
    "Net Income Common Stockholders", "Net Income",
    "Diluted EPS", "Basic EPS",
    "Selling General And Administration", "Research And Development",
]

# 现金流关键指标
cf_key = [
    "Free Cash Flow", "Operating Cash Flow",
    "Capital Expenditure", "Investing Cash Flow", "Financing Cash Flow",
]

# 透视每个sheet
def pivot_quarterly(df_long, key_items, value_col, prefix=""):
    """透视季度数据: 每行=股票+季度, 每列=一个指标"""
    sub = df_long[df_long["item"].isin(key_items)].copy()
    piv = sub.pivot_table(
        index=["symbol", "report_date"],
        columns="item",
        values=value_col,
        aggfunc="first"
    ).reset_index()
    # 列名清理
    piv.columns = [prefix + str(c).replace(" ", "_").lower() for c in piv.columns]
    if prefix:
        piv = piv.rename(columns={
            prefix + "symbol": "symbol",
            prefix + "report_date": "report_date"
        })
    return piv

df_bs_pivot = pivot_quarterly(df_bs_long, bs_key, "balance_value", "bs_")
df_is_pivot = pivot_quarterly(df_is_long, is_key, "income_value", "is_")
df_cf_pivot = pivot_quarterly(df_cf_long, cf_key, "cashflow_value", "cf_")

print(f"BS pivot: {df_bs_pivot.shape}")
print(f"IS pivot: {df_is_pivot.shape}")
print(f"CF pivot: {df_cf_pivot.shape}")

In [ ]:
# ==========================================
# 第6步: 合并三张表 -> 单一季度基本面数据
# ==========================================

df_q = df_bs_pivot.merge(df_is_pivot, on=["symbol", "report_date"], how="outer")
df_q = df_q.merge(df_cf_pivot, on=["symbol", "report_date"], how="outer")
df_q = df_q.sort_values(["symbol", "report_date"]).reset_index(drop=True)

print(f"合并后季度数据: {df_q.shape}")
print(f"日期范围: {df_q['report_date'].min()} ~ {df_q['report_date'].max()}")
print(f"\n列: {df_q.columns.tolist()}")

In [ ]:
# ==========================================
# 第7步: 保存
# ==========================================

df_q.to_csv(
    os.path.join(OUT_DIR, "day5_quarterly_fundamentals.csv"),
    index=False, encoding="utf-8-sig"
)

df_info.to_csv(
    os.path.join(OUT_DIR, "day5_current_valuation.csv"),
    index=False, encoding="utf-8-sig"
)

print("Day5 完成!")
print(f"  day5_quarterly_fundamentals.csv — {df_q.shape}")
print(f"  day5_current_valuation.csv — {df_info.shape}")